# March Machine Learning Mania 2026

|No| Name  | Notebook |
| --- | ------ | ------- |
| 00 | Dataset |  https://www.kaggle.com/competitions/march-machine-learning-mania-2026/data |
| 01 | Understand Data Structure  | https://www.kaggle.com/code/rudraprasadbhuyan/ncaa-p1-dataset-overview-structure-understanding/ |
| 02 | Create a baseline model only on Male rqd. 4 datasets | https://www.kaggle.com/code/rudraprasadbhuyan/ncaa-p2-basline-male-4-datasets/ |
| 03 | Features Eng | https://www.kaggle.com/code/rudraprasadbhuyan/ncaa-p3-features-eng/ |
| 04 | Simple ELO | https://www.kaggle.com/code/rudraprasadbhuyan/ncaa-p4-simple-elo/ |
| 05 | Adv ELO | https://www.kaggle.com/code/rudraprasadbhuyan/ncaa-p5-adv-elo/  |
| | |  |

In [3]:
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss

**Story**

Features

- Part 2 V1 - Baseline model (`win diff`, `seed diff`)
- Part 3 
  - V1 - `margin diff`
  - V2 - `recent win pct diff` Hypothesis: Teams playing well recently perform better than tournament
- Part 4
  - V1: Simple ELO + All Features
  - V2: Remove these 'Margin_Diff','RecentWinPct_Diff'still good result.
- Part 5
    -  V1: Add Home Advantage as a multiplier in the elo function

# Basic 

> - Again, without understanding the basic we can't code.
> - First, I am new to sports analysis.
> - So we need info and data to understand the game rules.
> - So if you new let's take the help of an infinitely intelligent


## What is ELO?

ELO is a dynamic rating system.

* Every team starts with same rating (ex: 1500)
* If strong team beats weak team → small rating change
* If weak team upsets strong team → big rating change

It automatically captures:

* strength
* upset power
* consistency
* schedule difficulty



## ELO Formula
- wiki: https://en.wikipedia.org/wiki/Elo_rating_system

For a match:

**Expected score:**

$$E_A = 1 / (1 + 10^{(R_B - R_A)/400}) $$

**Update:**

$$R_A = R_A + K * (S_A - E_A)$$
Where:

* `R_A` = rating of Team A
* `R_B` = rating of Team B
* `S_A` = 1 if win, 0 if loss
* `K` = learning rate (usually 20)

---


# Import Data

In [5]:
# Root file
data_file_path = r"/kaggle/input/march-machine-learning-mania-2026"
# Root file
data_file_path = r"C:\Users\Rudra\Desktop\kaggle\NCAA\data"

# Load only required files
regular = pd.read_csv(os.path.join(data_file_path, "MRegularSeasonCompactResults.csv"))
tourney = pd.read_csv(os.path.join(data_file_path,"MNCAATourneyCompactResults.csv"))
seeds = pd.read_csv(os.path.join(data_file_path,"MNCAATourneySeeds.csv"))
submission = pd.read_csv(os.path.join(data_file_path,"SampleSubmissionStage1.csv"))

# Sample View
display(regular.sample(3))
display(tourney.sample(3))
display(seeds.sample(3))
display(submission.sample(3))

,Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT
60172,1999,122,1285,87,1340,81,H,0
113564,2011,13,1442,59,1366,52,A,0
154170,2018,89,1317,69,1198,67,H,1


,Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT
2435,2023,139,1462,84,1338,73,N,0
853,1998,138,1268,67,1228,61,N,0
820,1998,136,1116,74,1304,65,N,0


,Season,Seed,TeamID
2252,2019,X16a,1192
2618,2025,Z09,1328
965,2000,W06,1231


,ID,Pred
353727,2024_3206_3395,0.5
455209,2025_3104_3449,0.5
512636,2025_3360_3410,0.5


# clean Datasets

In [6]:
wins = regular.groupby(['Season', 'WTeamID']).size().reset_index(name='wins')
losses = regular.groupby(['Season', 'LTeamID']).size().reset_index(name='losses')

wins = wins.rename(columns={"WTeamID": 'TeamID'})
losses = losses.rename(columns={'LTeamID': 'TeamID'})

team_stats = pd.merge(
    wins, losses,
    how="outer", on=["Season", 'TeamID']
).fillna(0)

team_stats["TotalGames"] = team_stats["wins"] + team_stats["losses"]
team_stats['WinPct'] = team_stats['wins'] / team_stats['TotalGames']

display(team_stats.sample(3))


# Clean Seeds
seeds['SeedNum'] = seeds['Seed'].str[1:3].astype(int)
display(seeds.sample(3))


# Clean Tournament
tourney['Team1'] = tourney[['WTeamID', 'LTeamID']].min(axis=1)
tourney['Team2'] = tourney[['WTeamID', 'LTeamID']].max(axis=1)
tourney['Team1Win'] = (tourney['WTeamID'] == tourney['Team1']).astype(int)

train = tourney[['Season', 'Team1', 'Team2', 'Team1Win']]
display(train.sample(3))
print(train.shape)

,Season,TeamID,wins,losses,TotalGames,WinPct
4260,1999,1245,21.0,6.0,27.0,0.777778
7867,2010,1227,7.0,22.0,29.0,0.241379
1442,1990,1108,7.0,18.0,25.0,0.280000


,Season,Seed,TeamID,SeedNum
210,1988,X03,1301,3
273,1989,X02,1231,2
1035,2001,W12,1429,12


,Season,Team1,Team2,Team1Win
2393,2023,1222,1297,1
1956,2015,1139,1323,0
702,1996,1280,1433,1


(2585, 4)


# Add Features

## ELO 


### Margin-Based ELO 

Right now:

Win by 1 point = Win by 30 points
That’s not realistic.

We modify K dynamically based on score difference.



### Margin Multiplier Formula

Common version:

$$ Multiplier = \log(|Margin| + 1) * \frac{2.2}{((R_w - R_l)*0.001 + 2.2)} $$

This means:

* Big win → bigger rating update
* Upset → bigger update
* Close win → smaller update


### ELO Calculator

In [15]:
def calculate_power_elo(df,
                        initial_rating=1500,
                        carry=0.75,
                        k_base=20,
                        home_advantage=80):

    df = df.sort_values(['Season', 'DayNum'])

    ratings = {}
    season_ratings = {}

    current_season = None

    for _, row in df.iterrows():

        season = row['Season']

        # Season change → regress toward mean
        if current_season is None:
            current_season = season

        if season != current_season:
            for team in ratings:
                ratings[team] = carry * ratings[team] + \
                                (1 - carry) * initial_rating
            current_season = season

        teamA = row['WTeamID']
        teamB = row['LTeamID']

        # Initialize if new
        if teamA not in ratings:
            ratings[teamA] = initial_rating
        if teamB not in ratings:
            ratings[teamB] = initial_rating

        RA = ratings[teamA]
        RB = ratings[teamB]

        # --- Dynamic K ---
        if row['DayNum'] < 50:
            k = k_base * 1.5
        elif row['DayNum'] > 120:
            k = k_base * 0.75
        else:
            k = k_base

        # --- Home advantage ---
        home_adv = 0
        if row['WLoc'] == 'H':
            home_adv = home_advantage
        elif row['WLoc'] == 'A':
            home_adv = -home_advantage

        RA_adj = RA + home_adv

        # --- Expected probability ---
        EA = 1 / (1 + 10 ** ((RB - RA_adj) / 400))
        EB = 1 - EA

        # --- Margin scaling ---
        margin = row['WScore'] - row['LScore']

        multiplier = np.log(abs(margin) + 1) * \
                     (2.2 / ((RA - RB) * 0.001 + 2.2))

        # --- Update ---
        ratings[teamA] = RA + k * multiplier * (1 - EA)
        ratings[teamB] = RB + k * multiplier * (0 - EB)

    # Save final rating snapshot per season
    for team, rating in ratings.items():
        season_ratings[(season, team)] = rating

    elo_df = pd.DataFrame(
        [(s, t, r) for (s, t), r in season_ratings.items()],
        columns=['Season','TeamID','ELO']
    )

    return elo_df

In [16]:
elo_ratings = calculate_power_elo(df=regular)

In [17]:
display(elo_ratings.sample(3))

,Season,TeamID,ELO
62,2026,1231,1737.606062
281,2026,1182,1581.340885
7,2026,1432,1499.996950


In [18]:
def add_feature_elo(df):
    # team1 elo
    df = df.merge(
        elo_ratings,
        left_on=['Season','Team1'],
        right_on=['Season','TeamID'],
        how='left'
    ).rename(columns={'ELO':'Team1_ELO'}).drop('TeamID', axis=1)

    # team2 elo
    df = df.merge(
        elo_ratings,
        left_on=['Season','Team2'],
        right_on=['Season','TeamID'],
        how='left'
    ).rename(columns={'ELO':'Team2_ELO'}).drop('TeamID', axis=1)
    
    return df

## Add Score Margin

In [19]:
regular['ScoreDiff'] = regular['WScore'] - regular['LScore']

# For Winners
w_margin = regular.groupby(['Season', 'WTeamID'])['ScoreDiff'].mean().reset_index()
w_margin = w_margin.rename(columns={'WTeamID':'TeamID', 'ScoreDiff':'AvgMargin'})

# For Losers
l_margin = regular.groupby(['Season','LTeamID'])['ScoreDiff'].mean().reset_index()
l_margin['ScoreDiff'] = -l_margin['ScoreDiff']
l_margin = l_margin.rename(columns={'LTeamID':'TeamID','ScoreDiff':'AvgMargin'})

# Combine
margin = pd.concat([w_margin, l_margin])
margin = margin.groupby(['Season','TeamID'])['AvgMargin'].mean().reset_index()
display(margin.sample(3))

,Season,TeamID,AvgMargin
6784,2007,1169,-1.063725
5977,2004,1376,3.106522
9021,2013,1350,2.099206


## Add Recent Win Pct

In [20]:
regular_recent =  regular[regular['DayNum'] > 100].copy()

# wins
wins_recent = regular_recent.groupby(['Season', 'WTeamID']).size().reset_index(name='Wins')
wins_recent = wins_recent.rename(columns={'WTeamID': 'TeamID'})

# Losses
losses_recent = regular_recent.groupby(['Season', 'LTeamID']).size().reset_index(name='Losses')
losses_recent = losses_recent.rename(columns={'LTeamID': 'TeamID'})

recent_stats = pd.merge(
    wins_recent, losses_recent,
    on=['Season', 'TeamID'],
    how='outer'
).fillna(0)

recent_stats['Total'] = recent_stats['Wins'] + recent_stats['Losses']
recent_stats['RecentWinPct'] =  recent_stats['Wins'] / recent_stats['Total']

recent_stats.head(3)


,Season,TeamID,Wins,Losses,Total,RecentWinPct
0,1985,1102,3.0,5.0,8.0,0.375000
1,1985,1103,3.0,4.0,7.0,0.428571
2,1985,1104,7.0,2.0,9.0,0.777778


In [21]:
def add_feature_recent_win_pct(df):
    # recent team 1 win pct
    df = df.merge(
        recent_stats[['Season','TeamID','RecentWinPct']],
        left_on=['Season','Team1'],
        right_on=['Season','TeamID'],
        how='left'
    ).rename(columns={'RecentWinPct':'Team1_RecentWinPct'}).drop('TeamID', axis=1)

    # recent team 2 win pct
    df = df.merge(
        recent_stats[['Season','TeamID','RecentWinPct']],
        left_on=['Season','Team2'],
        right_on=['Season','TeamID'],
        how='left'
    ).rename(columns={'RecentWinPct':'Team2_RecentWinPct'}).drop('TeamID', axis=1)


    return df

## Add Win Pct Diff 

In [22]:
def add_feature_win_pct(df:pd.date_range) -> pd.DataFrame:
    """Create features for both train and test(submission) data."""

    # Team1 win pct
    df = df.merge(
        team_stats[['Season','TeamID','WinPct']],
        left_on=['Season','Team1'],
        right_on=['Season','TeamID'],
        how='left'
    ).rename(columns={'WinPct':'Team1_WinPct'}).drop('TeamID', axis=1)

    # Team2 win pct
    df = df.merge(
        team_stats[['Season','TeamID','WinPct']],
        left_on=['Season','Team2'],
        right_on=['Season','TeamID'],
        how='left'
    ).rename(columns={'WinPct':'Team2_WinPct'}).drop('TeamID', axis=1)
    
    return df

## Add Seed

In [23]:

def add_feature_seed(df):
    # team1 seed
    df = df.merge(
        seeds[['Season','TeamID','SeedNum']],
        left_on=['Season','Team1'],
        right_on=['Season','TeamID'],
        how='left'
    ).rename(columns={'SeedNum':'Team1_Seed'}).drop('TeamID', axis=1)

    # team 2 seed
    df = df.merge(
        seeds[['Season','TeamID','SeedNum']],
        left_on=['Season','Team2'],
        right_on=['Season','TeamID'],
        how='left'
    ).rename(columns={'SeedNum':'Team2_Seed'}).drop('TeamID', axis=1)
    
    return df

## Add Avg Margin

In [24]:
def add_feature_avg_margin(df):
    # team 1 avg margin
    df = df.merge(
        margin,
        left_on=['Season', 'Team1'],
        right_on=['Season', 'TeamID'],
        how='left',
    ).rename(columns={'AvgMargin': 'Team1_AvgMargin'}).drop('TeamID', axis=1)
    
    # team 2 avg margin
    df = df.merge(
        margin,
        left_on=['Season', 'Team2'],
        right_on=['Season', 'TeamID'],
        how='left',
    ).rename(columns={'AvgMargin': 'Team2_AvgMargin'}).drop('TeamID', axis=1)

    return df

In [25]:
def add_all_features(df):
    """Call al the features function at a time."""
    df = add_feature_win_pct(df)
    df = add_feature_seed(df)
    # df = add_feature_avg_margin(df)
    # df = add_feature_recent_win_pct(df)
    df = add_feature_elo(df)
    
    return df

In [26]:
train = add_all_features(train)

# Create Model Inputs Features

In [ ]:
def create_diff(df:pd.DataFrame) ->pd.DataFrame:
    """ Create Features difference."""
    
    df['WinPct_Diff'] = df['Team1_WinPct'] - df['Team2_WinPct']

    df['Seed_Diff'] = df['Team2_Seed'] - df['Team1_Seed'] 

    # df['Margin_Diff'] = df['Team1_AvgMargin'] - df['Team2_AvgMargin']
    # df['RecentWinPct_Diff'] = df['Team1_RecentWinPct'] - df['Team2_RecentWinPct']

    df['ELO_Diff'] =  df['Team1_ELO'] - df['Team2_ELO']

    return df

In [28]:
train = create_diff(train)

train.sample(3)

,Season,Team1,Team2,Team1Win,Team1_WinPct,Team2_WinPct,Team1_Seed,Team2_Seed,Team1_ELO,Team2_ELO,WinPct_Diff,Seed_Diff,ELO_Diff
1440,2007,1400,1425,0,0.727273,0.676471,4,5,NaN,NaN,0.050802,1,NaN
1461,2008,1277,1396,1,0.757576,0.636364,5,12,NaN,NaN,0.121212,7,NaN
624,1994,1181,1345,1,0.821429,0.866667,2,1,NaN,NaN,-0.045238,-1,NaN


# Model

In [29]:
# model_train_input_cols = ['WinPct_Diff','Seed_Diff','Margin_Diff','RecentWinPct_Diff', 'ELO_Diff']
model_train_input_cols = ['WinPct_Diff','Seed_Diff', 'ELO_Diff']

In [33]:
train_data = train[train['Season'] < 2024].fillna(0)
val_data   = train[train['Season'] == 2024].fillna(0)

X_train = train_data[model_train_input_cols]
y_train = train_data['Team1Win']

X_val = val_data[model_train_input_cols]
y_val = val_data['Team1Win']

model = LogisticRegression(max_iter=5000)
model.fit(X_train, y_train)

preds = model.predict_proba(X_val)[:, 1]

print(f'LogLoss {log_loss(y_val, preds)}')

LogLoss 0.5661304594703703


# Submission

In [34]:
def create_id(df):
    df[['Season','Team1','Team2']] = df['ID'].str.split('_', expand=True)

    df['Season'] = df['Season'].astype(int)
    df['Team1'] = df['Team1'].astype(int)
    df['Team2'] = df['Team2'].astype(int)

    return df

submission = create_id(submission)
submission = add_all_features(submission)

# only men for now
submission_men = submission[
    (submission['Team1'] < 2000) &
    (submission['Team2'] < 2000)
].copy()

submission = create_diff(submission)

submission[model_train_input_cols] = submission[model_train_input_cols].fillna(0)

# Predict
submission['Pred'] = model.predict_proba(
    submission[model_train_input_cols]
)[:,1]

display(submission[['ID','Pred']])

# Export
submission[['ID','Pred']].to_csv("submission.csv", index=False)
print('submission.csv exported')

ValueError: Columns must be same length as key